In [ ]:
import re
import pandas as pd
from collections import defaultdict

def analyze_ns3_output(log_file_path):
    """
    Parses the ns3 log file to track packet paths, timings, and tags.

    Args:
        log_file_path (str): The path to the ns3 log file.

    Returns:
        pandas.DataFrame: A DataFrame with packet information, paths, and timings.
    """
    packets_info = {}
    packet_hops = defaultdict(list)

    # Regex to parse FlowInfo and LinkTx lines, now capturing more data
    flow_info_re = re.compile(r"FlowInfo, .*PktUID: (\d+), OriginalSrc: (\d+), FinalDst: (\d+), Sport: (\d+), FlowTag: (\d+)")
    flow_info_1_re = re.compile(r"FlowInfo-1, PktUID: (\d+)")
    link_tx_re = re.compile(r"LinkTx, Timestamp: (\d+)ns, .*PktUID: (\d+), LinkSrc: (\d+)")

    with open(log_file_path, 'r') as f:
        for line in f:
            # Match FlowInfo lines to get packet metadata
            flow_match = flow_info_re.search(line)
            if flow_match:
                pkt_uid, src, dst, port, tag = map(int, flow_match.groups())
                if pkt_uid not in packets_info:
                    packets_info[pkt_uid] = {'OriginalSrc': src, 'FinalDst': dst, 'Tag': tag, 'Port': port}
                continue

            # Match FlowInfo-1 lines (no OriginalSrc/Tag)
            flow_1_match = flow_info_1_re.search(line)
            if flow_1_match:
                pkt_uid = int(flow_1_match.group(1))
                if pkt_uid not in packets_info:
                    packets_info[pkt_uid] = {'OriginalSrc': None, 'FinalDst': None, 'Tag': None}
                continue

            # Match LinkTx lines to build the packet path with timestamps
            link_match = link_tx_re.search(line)
            if link_match:
                timestamp, pkt_uid, link_src = map(int, link_match.groups())
                # Append (source, timestamp) to the packet's hop list
                current_hops = packet_hops[pkt_uid]
                if not current_hops or current_hops[-1][0] != link_src:
                    current_hops.append((link_src, timestamp))

    # Combine the parsed data into a list of dictionaries
    records = []
    for uid, hops in packet_hops.items():
        info = packets_info.get(uid, {})
        path_nodes = [hop[0] for hop in hops]
        
        start_time = hops[0][1] if hops else 0
        end_time = hops[-1][1] if hops else 0
        delay = end_time - start_time if hops else 0
        
        original_src = info.get('OriginalSrc')
        if original_src is None and path_nodes:
            original_src = path_nodes[0]
        
        records.append({
            'PktUID': uid,
            'Tag': info.get('Tag'),
            'Port' : info.get('Port'),
            'OriginalSrc': original_src,
            'FinalDst': info.get('FinalDst'),
            'StartTime (ns)': start_time,
            'EndTime (ns)': end_time,
            'Delay (ns)': delay,
            'Path': ' -> '.join(map(str, path_nodes)),
            'Path_List': path_nodes
        })

    # Create and return a DataFrame
    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values(by=['OriginalSrc', 'PktUID']).reset_index(drop=True)
    return df

# --- Main execution ---
# Path to your ns3 log file
ns3_log_path = "/home/xavid/feina/astra-sim/upc/text_ns3.txt"

# Analyze the log file
packet_tracking_df = analyze_ns3_output(ns3_log_path)

# Display the paths for packets originating from each source node
if not packet_tracking_df.empty:
    for src_node in sorted(packet_tracking_df['OriginalSrc'].dropna().unique()):
        print(f"--- Paths from OriginalSrc: {src_node} ---")
        src_df = packet_tracking_df[packet_tracking_df['OriginalSrc'] == src_node]
        
        # Group by path to show unique paths and the packets that took them
        path_groups = src_df.groupby('Path')['PktUID'].apply(list)
        for path, uids in path_groups.items():
            print(f"  Path: {path}")
            print(f"    PktUIDs: {uids}\n")
else:
    print("No packet information found in the log file.")



In [ ]:
if not packet_tracking_df.empty:
    # Get unique pairs of OriginalSrc and FinalDst
    unique_pairs = packet_tracking_df[['OriginalSrc', 'FinalDst']].dropna().drop_duplicates()
    
    for index, row in unique_pairs.iterrows():
        src = int(row['OriginalSrc'])
        dst = int(row['FinalDst'])
        
        print(f"--- Path counts from OriginalSrc: {src} to FinalDst: {dst} ---")
        
        # Filter for the specific src-dst pair
        filtered_df = packet_tracking_df[
            (packet_tracking_df['OriginalSrc'] == src) & 
            (packet_tracking_df['FinalDst'] == dst)
        ]
        
        # Count the different paths taken
        path_counts = filtered_df['Path_List'].value_counts()
        
        if not path_counts.empty:
            print(path_counts)
        else:
            print("No paths found for this pair.")
        print("-" * 50)
else:
    print("Packet tracking DataFrame is empty.")

In [ ]:
packet_tracking_df[(packet_tracking_df['OriginalSrc'] == 2) ].head()

In [ ]:
if not packet_tracking_df.empty:
    # Get unique pairs of OriginalSrc and FinalDst
    unique_pairs = packet_tracking_df[['OriginalSrc', 'FinalDst']].dropna().drop_duplicates()
    
    for index, row in unique_pairs.iterrows():
        src = int(row['OriginalSrc'])
        dst = int(row['FinalDst'])
        
        print(f"--- Sequential Path Analysis for OriginalSrc: {src} to FinalDst: {dst} ---")
        
        # Filter for the specific src-dst pair, already sorted by PktUID
        pair_df = packet_tracking_df[
            (packet_tracking_df['OriginalSrc'] == src) & 
            (packet_tracking_df['FinalDst'] == dst)
        ].copy()
        
        if not pair_df.empty:
            # Identify consecutive groups of packets taking the same path
            # A new group starts when the path changes from the previous packet
            pair_df['PathStr'] = pair_df['Path_List'].astype(str)
            pair_df['PathGroup'] = (pair_df['PathStr'] != pair_df['PathStr'].shift()).cumsum()

            # Group by these sequential path groups and aggregate the results
            grouped_paths = pair_df.groupby('PathGroup').agg(
                Path=('Path', 'first'),
                PacketCount=('PktUID', 'count'),
                GroupStartTime=('StartTime (ns)', 'min'),
                GroupEndTime=('EndTime (ns)', 'max')
            )
            
            # Calculate duration for each group
            grouped_paths['GroupDuration (ns)'] = grouped_paths['GroupEndTime'] - grouped_paths['GroupStartTime']
            
            # Display the results for each sequential group
            for i, group_row in grouped_paths.iterrows():
                print(f"\n  Path: {group_row['Path']}")
                print(f"    Sequential Packet Count: {group_row['PacketCount']}")
                print(f"    Group Start Time:        {group_row['GroupStartTime']} ns")
                print(f"    Group End Time:          {group_row['GroupEndTime']} ns")
                print(f"    Group Duration:          {group_row['GroupDuration (ns)']} ns")
        else:
            print("  No paths found for this pair.")
        print("\n" + "-" * 70)
else:
    print("Packet tracking DataFrame is empty.")

In [ ]:
pd.set_option('display.max_rows', 600)

In [ ]:
packet_tracking_df[(packet_tracking_df['OriginalSrc'] == 2)& (packet_tracking_df['FinalDst'] == 4)].head(50)

In [ ]:
import pandas as pd
from synthesize_results import synthesize
def process_and_sort_results(df):
    """Process the DataFrame to merge ns3 results and sort."""
    if df.empty:
        return df
    
    # Drop columns that are not needed for joining
    df = df.drop(columns=['tag', 'workload_node_id'])
    
    # Group by the message identifiers and aggregate the data
    # This will merge rows for ns3 with the other models
    group_cols = ['src', 'dst', 'chunk_id']
    df = df.groupby(group_cols).sum().reset_index()
    
    # Sort by g2 start time if the column exists
    if 'start_time_g2' in df.columns:
        df = df.sort_values(by='start_time_g2').reset_index(drop=True)
        
    return df

# List of run folders to analyze
run_folders = [
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_101154",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_101517",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_101856",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_102253",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_102624",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_102931",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_103301",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_103615",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_103924",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_104213",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_141336",
    "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251022_101351"
    # Add other run folder paths here
]

all_results = []
for folder in run_folders:
    # Synthesize results for each run
    results_df = synthesize(folder)
    # Process and sort the DataFrame for the current run
    processed_df = process_and_sort_results(results_df)
    all_results.append(processed_df)

# Concatenate all results into a single DataFrame
if all_results:
    combined_df = pd.concat(all_results)
    
    # Group by the identifiers and calculate the mean across all runs
    group_cols = ['src', 'dst', 'chunk_id']
    averaged_df = combined_df.groupby(group_cols).mean().reset_index()
    
    # Optionally, sort the final averaged DataFrame
    if 'start_time_g2' in averaged_df.columns:
        averaged_df = averaged_df.sort_values(by='start_time_g2').reset_index(drop=True)
else:
    averaged_df = pd.DataFrame()


# Muestra el DataFrame resultante
averaged_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Filter the DataFrame for chunk_id == 2 (accept int or str)
chunk_2_df = averaged_df[averaged_df['chunk_id'].isin([2, '2'])]

# Check if the filtered DataFrame is not empty and contains the required columns
if not chunk_2_df.empty and 'arrival_time_g2' in chunk_2_df.columns and 'arrival_time_ns3' in chunk_2_df.columns:
    data = chunk_2_df[['arrival_time_g2', 'arrival_time_ns3']].dropna().astype(float)
    x = data['arrival_time_g2']
    y = data['arrival_time_ns3']

    # compute common symmetric limits with a small margin
    vmin = min(x.min(), y.min())
    vmax = max(x.max(), y.max())
    if np.isfinite(vmin) and np.isfinite(vmax):
        span = vmax - vmin
        if span == 0:
            margin = 0.1 * (abs(vmin) if vmin != 0 else 1.0)
        else:
            margin = 0.05 * span
        lims = (vmin - margin, vmax + margin)
    else:
        lims = (0, 1)

    plt.figure(figsize=(8, 8))
    plt.scatter(x, y, alpha=0.7, label='Chunk 2 Arrivals')
    plt.plot(lims, lims, 'r--', alpha=0.75, zorder=0, label='Ideal Match (y=x)')

    plt.xlim(lims)
    plt.ylim(lims)

    ax = plt.gca()
    ax.set_aspect('equal', adjustable='box')

    plt.xlabel("Arrival Time G2 (ns)")
    plt.ylabel("Arrival Time NS3 (ns)")
    plt.title("Comparison of Arrival Times (G2 vs. NS3) for Chunk ID 2")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("No data to plot for chunk_id=2, or 'arrival_time_g2'/'arrival_time_ns3' columns are missing.")